# Generate per-class samples across DiffiT 256² training checkpoints

Generates `SAMPLES_PER_CLASS` images for each class at five training checkpoints from `00017-diffit-256-gpus2-batch192`. Saves output as one `.npz` per `(checkpoint, class)` so the loop is resumable.

**Requested kimg → closest available snapshot:**

| Requested kimg | Snapshot file                 |
|---------------:|:------------------------------|
| 4500           | `network-snapshot-004435.pt` |
| 6300           | `network-snapshot-006451.pt` |
| 8000           | `network-snapshot-008064.pt` |
| 12000          | `network-snapshot-012096.pt` |
| 16000          | `network-snapshot-016128.pt` |

(Snapshots are spaced ~403 kimg apart, so exact matches don't exist.)

**Compute scope:** With `SAMPLES_PER_CLASS=10000` and `num_classes=3`, this is 30K samples per checkpoint (150K total across the five). The loop saves one `.npz` per `(checkpoint, class)` and skips files that already exist, so it's safely resumable.

## 1 — Imports and path setup

In [ ]:
import os, sys, time
from pathlib import Path

NB_DIR = Path.cwd()
REPO_ROOT = NB_DIR.parent.parent          # .../DiffiT-v2
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import torch
from tqdm.auto import tqdm
from diffusers.models import AutoencoderKL

import diffit.diffit as diffit_module
from diffit import create_diffusion, diffusion_defaults

print(f'REPO_ROOT: {REPO_ROOT}')
print(f'torch {torch.__version__}, CUDA available: {torch.cuda.is_available()}')

## 2 — Configuration

All knobs in one place. The `CHECKPOINT_KIMGS` list is the *closest available* snapshot for each requested kimg — see the table at the top of this notebook.

In [ ]:
RUN_DIR = Path('/home/david/mnt/ssd_2_sata/python/phd/DiffiT-v2/training-runs/00017-diffit-256-gpus2-batch192')

# kimg → snapshot filename id (closest available to {4500, 6300, 8000, 12000, 16000})
CHECKPOINT_KIMGS = [4435, 6451, 8064, 12096, 16128]

SAMPLES_PER_CLASS  = 10   # per (checkpoint, class)
BATCH_SIZE         = 32      # samples per forward pass; tune to GPU
NUM_SAMPLING_STEPS = 250
CFG_SCALE          = 4.4
USE_DDIM           = False
SCALE_POW          = 4.0     # cosine CFG schedule power for 256² (matches gen_images.py)
VAE_DECODER        = 'ema'   # 'ema' | 'mse'
BASE_SEED          = 0       # seed = BASE_SEED + kimg*10**6 + class*10**3 + batch_idx

OUT_DIR = REPO_ROOT / 'experiments' / 'samples' / '00017-diffit-256-gpus2-batch192'
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cuda.matmul.allow_tf32 = True

# Verify all snapshots exist before starting a long run.
for k in CHECKPOINT_KIMGS:
    p = RUN_DIR / f'network-snapshot-{k:06d}.pt'
    assert p.exists(), f'Missing checkpoint: {p}'
    print(f'  ok  {p.name}')
print(f'\nOutput dir: {OUT_DIR}')
print(f'Device: {DEVICE}')

## 3 — Load the VAE decoder (once)

The VAE is checkpoint-independent, so we keep it resident across all five model loads.

In [ ]:
vae = AutoencoderKL.from_pretrained(f'stabilityai/sd-vae-ft-{VAE_DECODER}').to(DEVICE).eval()
for p in vae.parameters():
    p.requires_grad_(False)

diff_config = diffusion_defaults()
diff_config['timestep_respacing'] = str(NUM_SAMPLING_STEPS)
diffusion = create_diffusion(**diff_config)
DIFFUSION_STEPS = diff_config['diffusion_steps']
print(f'VAE + diffusion ready (steps={NUM_SAMPLING_STEPS}, base_steps={DIFFUSION_STEPS})')

## 4 — Helpers

`load_diffit_model` follows the same EMA-loading logic as `experiments/analyze_sample_split.py::load_model` and infers `num_classes` from the embedding table.

`generate_class_batch` mirrors the CFG sampling block in `gen_images.py`.

In [ ]:
def load_diffit_model(ckpt_path, device, model_name='Diffit'):
    ckpt = torch.load(ckpt_path, map_location='cpu')
    if isinstance(ckpt, dict) and 'ema' in ckpt:
        state, image_size = ckpt['ema'], ckpt.get('image_size', 256)
    elif isinstance(ckpt, dict) and 'model' in ckpt:
        state, image_size = ckpt['model'], ckpt.get('image_size', 256)
    else:
        state, image_size = ckpt, 256

    # +1 slot is the CFG null token
    num_classes = state['y_embedder.embedding_table.weight'].shape[0] - 1
    latent_size = image_size // 8

    model = diffit_module.__dict__[model_name](
        input_size=latent_size, num_classes=num_classes,
    )
    missing, unexpected = model.load_state_dict(state, strict=False)
    if missing or unexpected:
        print(f'  load_state_dict: missing={len(missing)} unexpected={len(unexpected)}')
    model.to(device).eval()
    return model, image_size, num_classes

In [ ]:
@torch.inference_mode()
def generate_class_batch(model, class_idx, batch_size, latent_size, num_classes, seed):
    """Generate a batch of `batch_size` images for a single class. Returns uint8 NHWC numpy."""
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

    z = torch.randn(batch_size, 4, latent_size, latent_size, device=DEVICE)
    classes      = torch.full((batch_size,), class_idx,    device=DEVICE, dtype=torch.long)
    classes_null = torch.full((batch_size,), num_classes,  device=DEVICE, dtype=torch.long)

    z_cfg = torch.cat([z, z], 0)
    model_kwargs = {
        'y':               torch.cat([classes, classes_null], 0),
        'cfg_scale':       CFG_SCALE,
        'diffusion_steps': DIFFUSION_STEPS,
        'scale_pow':       SCALE_POW,
    }

    sample_fn = diffusion.ddim_sample_loop if USE_DDIM else diffusion.p_sample_loop
    sample = sample_fn(
        model.forward_with_cfg,
        z_cfg.shape, z_cfg,
        clip_denoised=False, progress=False,
        model_kwargs=model_kwargs, device=DEVICE,
    )
    sample, _ = sample.chunk(2, dim=0)

    sample = vae.decode(sample / 0.18215).sample
    sample = ((sample + 1) * 127.5).clamp(0, 255).to(torch.uint8)
    return sample.permute(0, 2, 3, 1).contiguous().cpu().numpy()

## 5 — Generation loop

Output layout:

```
OUT_DIR/
  kimg_004435/
    class_0000.npz  # imgs: uint8 (SAMPLES_PER_CLASS, 256, 256, 3),  label: int
    class_0001.npz
    ...
  kimg_006451/
  ...
```

Per-class files exist already are skipped, so the cell is safely re-runnable. Set `CLASS_FILTER` to a list of ints to generate only a subset of classes (useful for testing).

In [ ]:
CLASS_FILTER = None   # e.g. [0, 1, 2] for a smoke test, or None for all classes

for kimg in CHECKPOINT_KIMGS:
    ckpt_path = RUN_DIR / f'network-snapshot-{kimg:06d}.pt'
    print(f'\n=== kimg={kimg}  ({ckpt_path.name}) ===')
    t_load = time.time()
    model, image_size, num_classes = load_diffit_model(str(ckpt_path), DEVICE)
    latent_size = image_size // 8
    print(f'  loaded in {time.time() - t_load:.1f}s  '
          f'image_size={image_size}  num_classes={num_classes}  latent_size={latent_size}')

    ckpt_outdir = OUT_DIR / f'kimg_{kimg:06d}'
    ckpt_outdir.mkdir(parents=True, exist_ok=True)

    classes = list(range(num_classes)) if CLASS_FILTER is None else list(CLASS_FILTER)

    for cls in tqdm(classes, desc=f'kimg={kimg}'):
        cls_path = ckpt_outdir / f'class_{cls:04d}.npz'
        if cls_path.exists():
            continue

        chunks = []
        n_done = 0
        batch_idx = 0
        while n_done < SAMPLES_PER_CLASS:
            bs = min(BATCH_SIZE, SAMPLES_PER_CLASS - n_done)
            seed = BASE_SEED + kimg * 10**6 + cls * 10**3 + batch_idx
            imgs = generate_class_batch(model, cls, bs, latent_size, num_classes, seed)
            chunks.append(imgs)
            n_done    += bs
            batch_idx += 1

        all_imgs = np.concatenate(chunks, axis=0)
        # Atomic-ish write: tmp file then rename, so a kill mid-write doesn't leave a half file.
        # tmp name ends in .npz so np.savez doesn't auto-append another '.npz'.
        tmp_path = ckpt_outdir / f'class_{cls:04d}.tmp.npz'
        np.savez(tmp_path, imgs=all_imgs, label=np.int64(cls))
        os.replace(tmp_path, cls_path)

    del model
    torch.cuda.empty_cache()
    print(f'  done kimg={kimg}')

print('\nAll checkpoints done.')

## 6 — Quick sanity check (optional)

Load one batch from one class file and display a 4×4 grid to verify the saved arrays look like images of the expected class.

In [ ]:
import matplotlib.pyplot as plt

PEEK_KIMG  = CHECKPOINT_KIMGS[-1]   # most-trained checkpoint
PEEK_CLASS = 0                      # one of the 3 classes

peek_path = OUT_DIR / f'kimg_{PEEK_KIMG:06d}' / f'class_{PEEK_CLASS:04d}.npz'
if peek_path.exists():
    data = np.load(peek_path)
    imgs = data['imgs']
    print(f'{peek_path.name}: imgs shape={imgs.shape}, dtype={imgs.dtype}, label={int(data["label"])}')

    fig, axes = plt.subplots(4, 4, figsize=(10, 10))
    for ax, img in zip(axes.flat, imgs[:16]):
        ax.imshow(img)
        ax.axis('off')
    fig.suptitle(f'kimg={PEEK_KIMG}  class={PEEK_CLASS}')
    fig.tight_layout()
    plt.show()
else:
    print(f'No file yet at {peek_path}; run cell 5 first or pick an earlier (kimg, class).')